# Fine-tuning a Frontier Model

Based on Week 6 Day 5: Fine-tune\ing GPT-4.1-nano on the Amazon pricing dataset.

In [ ]:
import os
import sys
import re
import json
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parents[1]))
from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

openai = OpenAI()

In [ ]:
LITE_MODE = True
TRAIN_SIZE = 1000
VAL_SIZE = 100
# MODEL = "gpt-4.1-nano-2025-04-14"
MODEL = "ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DHAXMjSo"
N_EPOCHS = 3
BATCH_SIZE = 4
SUFFIX = "pricer"
SEED = 42

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
fine_tune_train = train[:TRAIN_SIZE]
fine_tune_validation = val[:VAL_SIZE]

print(f"Fine-tune train: {len(fine_tune_train)}, validation: {len(fine_tune_validation)}")

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

messages_for(fine_tune_train[0])

In [ ]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

print(make_jsonl(fine_tune_train[:3]))

In [ ]:
jsonl_dir = Path("jsonl")
jsonl_dir.mkdir(exist_ok=True)

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

write_jsonl(fine_tune_train, jsonl_dir / "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, jsonl_dir / "fine_tune_validation.jsonl")
print("JSONL files written.")

In [ ]:
with open(jsonl_dir / "fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open(jsonl_dir / "fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

print(f"Train file: {train_file.id}")
print(f"Validation file: {validation_file.id}")

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=MODEL,
    seed=SEED,
    hyperparameters={"n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE},
    suffix=SUFFIX,
)

job_id = job.id
print(f"Job created: {job_id}")

In [72]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-Cl6SXtI7d7wwhPtWPEQDctdi', created_at=1772983375, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DHAXMjSo', finished_at=1772984450, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=0.1, n_epochs=3), model='ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DGNpDZWT', object='fine_tuning.job', organization_id='org-nD8IhZgwkEEytjCy88NfYPb5', result_files=['file-2dRps8dhijSpDJSnThQ5kA'], seed=42, status='succeeded', trained_tokens=338136, training_file='file-KFPDyeNeF379nTWs7LFNPF', validation_file='file-BDyVnPpm61Z8oLneC62G1D', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=4, learning_rate_multiplier=0.1, n_epochs=3))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=True, eval_id=None, internal_worker_bac

In [73]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-hNNjZRXsR7iOsANEbMetAW5G', created_at=1772985180, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-NX5AJf8NxhDPWdVGz7dLeLVU', created_at=1772985178, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-tvRQ6JirOl8XRvdY3bkoR6vL', created_at=1772985178, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DHAXMjSo passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blo

You can also monitor your job in the OpenAI dashboard: https://platform.openai.com/finetune

Re-run the two cells above to check progress. Once the status is `succeeded`, continue below.

In [74]:
job = openai.fine_tuning.jobs.retrieve(job_id)
if job.status == "succeeded":
    fine_tuned_model_name = job.fine_tuned_model
    print(f"Fine-tuned model: {fine_tuned_model_name}")
else:
    print(f"Job status: {job.status}")
    if job.status == "running":
        print("Re-run this cell once the job has succeeded to get the model name.")
    elif job.status == "failed" and job.error:
        print(f"Error: {job.error.message}")
    if job.fine_tuned_model:
        fine_tuned_model_name = job.fine_tuned_model
        print(f"Model: {fine_tuned_model_name}")

Fine-tuned model: ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DHAXMjSo


In [75]:
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [77]:
def gpt_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return response.choices[0].message.content

In [78]:
print(f"Actual:    ${test[0].price:.2f}")
print(f"Predicted: {gpt_fine_tuned(test[0])}")

Actual:    $219.00
Predicted: $179.00


In [79]:
evaluate(gpt_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$30 $163 $26 $30 $63 $19 $64 $53 $3 $141 $442 $40 $16 $26 $36 $10 $31 $12 $163 $30 $10 $46 $12 $55 $198 $224 $64 $0 $140 $60 $0 $13 $113 $45 $65 $340 $136 $29 $38 $29 $169 $52 $0 $5 $101 $15 $28 $12 $68 $34 $20 $50 $172 $20 $47 $10 $7 $90 $34 $7 $138 $28 $5 $29 $199 $7 $20 $275 $49 $7 $19 $14 $87 $4 $40 $34 $142 $2 $0 $5 $31 $19 $49 $69 $3 $20 $58 $131 $50 $17 $4 $30 $4 $17 $2 $138 $6 $493 $75 $159 $26 $43 $32 $34 $70 $50 $18 $345 $14 $127 $0 $320 $57 $27 $24 $100 $15 $1 $24 $131 $18 $400 $19 $415 $70 $129 $16 $66 $2 $35 $9 $46 $5 $10 $101 $1 $118 $10 $59 $21 $15 $1 $3 $8 $104 $58 $20 $20 $2 $8 $2 $114 $25 $75 $6 $35 $61 $36 $459 $13 $91 $17 $24 $1 $160 $10 $202 $30 $5 $9 $39 $14 $210 $27 $15 $13 $2 $32 $176 $11 $136 $23 $233 $64 $25 $8 $71 $12 $10 $20 $64 $23 $2 $33 $54 $70 $29 $18 $2 $4 

## Reference scores

| Config | Error |
|---|---|
| nano 200 examples | $96.58 |
| mini 200 examples | $96.58 |
| mini 2,000 examples | $79.29 |
| nano 2,000 examples | $82.26 |
| nano 20,000 examples | $67.75 |